# Task 14: Custom CUDA Kernel Integration for Accelerated Activation Functions


## Objective

Compile a custom PyTorch CUDA extension implementing a vectorized SwiGLU activation and compare it with a PyTorch baseline.


## Short Theory

SwiGLU uses SiLU(x) multiplied by a gate. A CUDA kernel can evaluate this element-wise across GPU threads.


## Step 1: Imports


In [1]:
import torch
from torch.utils.cpp_extension import load_inline
print("CUDA available:", torch.cuda.is_available())


CUDA available: False


## Step 2: CUDA Extension


In [2]:
cpp = r'''
#include <torch/extension.h>
torch::Tensor swiglu_cuda(torch::Tensor x, torch::Tensor gate);
'''
cuda = r'''
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>
__global__ void k(const float* x,const float* g,float* o,int n){
  int i=blockIdx.x*blockDim.x+threadIdx.x;
  if(i<n){ float s=x[i]/(1.0f+expf(-x[i])); o[i]=s*g[i]; }
}
torch::Tensor swiglu_cuda(torch::Tensor x, torch::Tensor g){
  auto o=torch::empty_like(x); int n=x.numel();
  k<<<(n+255)/256,256>>>(x.data_ptr<float>(),g.data_ptr<float>(),o.data_ptr<float>(),n);
  return o;
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME,m){m.def("swiglu_cuda",&swiglu_cuda);}
'''
if torch.cuda.is_available():
    ext = load_inline("swiglu_ext", cpp_sources=cpp, cuda_sources=cuda, functions=None)
    print("CUDA extension compiled.")
else:
    print("CUDA not available; compile this notebook on a CUDA machine.")


CUDA not available; compile this notebook on a CUDA machine.


## Step 3: Benchmark


In [3]:
if torch.cuda.is_available():
    x = torch.randn(1_000_000, device="cuda")
    g = torch.randn_like(x)
    y = ext.swiglu_cuda(x,g)
    print("Output:", y.shape)


## Small Experiment

Correctness Check


In [4]:
if torch.cuda.is_available():
    ref = torch.nn.functional.silu(x)*g
    print("Max error:", (ref-y).abs().max().item())


## Conclusion

Provided the required C++/CUDA extension structure for a vectorized SwiGLU kernel and a correctness/benchmark hook.
